In [2]:
import pandas as pd

# ------------------------- STEP 1: LOAD DATA -------------------------

# Load patient demographics
patients = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/patients.csv")

# Load hospital admissions
admissions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/admissions.csv")

# Load ICU stays
icustays = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/icu/icustays.csv")

# Load diagnoses (ICD codes assigned to patients)
diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/diagnoses_icd.csv")

# Load prescriptions (medications administered)
prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")

# Load medical procedures (ICD codes for treatments)
procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/procedures_icd.csv")

# Load ICD descriptions (for both ICD-9 and ICD-10)
icd_diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_diagnoses.csv.gz")
icd_procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_procedures.csv.gz")

# Convert ICU stay timestamps to datetime format
icustays['intime'] = pd.to_datetime(icustays['intime'], errors='coerce')
icustays['outtime'] = pd.to_datetime(icustays['outtime'], errors='coerce')


# ------------------------- STEP 2: MAP ICD CODES TO TEXT DESCRIPTIONS -------------------------

# Merge diagnoses with descriptions
diagnoses = diagnoses.merge(icd_diagnoses, on="icd_code", how="left")
diagnoses['diagnosis_description'] = diagnoses['long_title'].fillna("Unknown diagnosis")

# Merge procedures with descriptions
procedures = procedures.merge(icd_procedures, on="icd_code", how="left")
procedures['procedure_description'] = procedures['long_title'].fillna("Unknown procedure")

/tmp/ipykernel_5389/3649365453.py:18: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")


In [3]:
# ------------------------- STEP 3: COMPUTE AGE & MERGE DATA -------------------------

# Keep only relevant patient information
patients = patients[['subject_id', 'anchor_age', 'anchor_year', 'gender']]

# Merge patient age & demographics into event tables
admissions = admissions.merge(patients, on='subject_id', how='left', suffixes=None)
icustays = icustays.merge(patients, on='subject_id', how='left', suffixes=None)
diagnoses = diagnoses.merge(patients, on='subject_id', how='left', suffixes=None)
procedures = procedures.merge(patients, on='subject_id', how='left', suffixes=None)
prescriptions = prescriptions.merge(patients, on='subject_id', how='left', suffixes=None)

# Convert date columns to datetime
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
icustays['intime'] = pd.to_datetime(icustays['intime'])
prescriptions['starttime'] = pd.to_datetime(prescriptions['starttime'])

# Compute patient age at each event
admissions['age_at_event'] = admissions['anchor_age'] + (admissions['admittime'].dt.year - admissions['anchor_year'])
icustays['age_at_event'] = icustays['anchor_age'] + (icustays['intime'].dt.year - icustays['anchor_year'])
diagnoses['age_at_event'] = diagnoses['anchor_age']
procedures['age_at_event'] = procedures['anchor_age']
prescriptions['age_at_event'] = prescriptions['anchor_age']

/tmp/ipykernel_5389/1067100215.py:7: FutureWarning: Passing 'suffixes' as a <class 'NoneType'>, is not supported and may give unexpected results. Provide 'suffixes' as a tuple instead. In the future a 'TypeError' will be raised.
  admissions = admissions.merge(patients, on='subject_id', how='left', suffixes=None)
/tmp/ipykernel_5389/1067100215.py:8: FutureWarning: Passing 'suffixes' as a <class 'NoneType'>, is not supported and may give unexpected results. Provide 'suffixes' as a tuple instead. In the future a 'TypeError' will be raised.
  icustays = icustays.merge(patients, on='subject_id', how='left', suffixes=None)
/tmp/ipykernel_5389/1067100215.py:9: FutureWarning: Passing 'suffixes' as a <class 'NoneType'>, is not supported and may give unexpected results. Provide 'suffixes' as a tuple instead. In the future a 'TypeError' will be raised.
  diagnoses = diagnoses.merge(patients, on='subject_id', how='left', suffixes=None)
/tmp/ipykernel_5389/1067100215.py:10: FutureWarning: Passing 

In [10]:
# ------------------------- STEP 4: BUILD A TABULAR DATASET -------------------------

# Aggregate data per hospitalization (`hadm_id`)
features = admissions[['subject_id', 'hadm_id', 'age_at_event', 'gender', 'admission_type', 'discharge_location']]

# ICU stays: Number of ICU admissions per hospitalization
# icu_summary = icustays.groupby('hadm_id').agg(
#     icu_admissions=('stay_id', 'count'),
#     icu_hours=('intime', lambda x: (x.max() - x.min()).total_seconds() / 3600)
# ).reset_index()

icu_summary = icustays.assign(
    icu_duration_hours=(icustays['outtime'] - icustays['intime']).dt.total_seconds() // 3600
).groupby('hadm_id').agg(
    icu_admissions=('stay_id', 'count'),
    total_icu_hours=('icu_duration_hours', 'sum')
).reset_index()

# Diagnoses: Count number of diagnoses per hospitalization
diagnosis_summary = diagnoses.groupby('hadm_id').agg(
    num_diagnoses=('icd_code', 'count'),
    unique_diagnoses=('diagnosis_description', lambda x: list(x.unique()))
).reset_index()

# Procedures: Count number of procedures per hospitalization
procedure_summary = procedures.groupby('hadm_id').agg(
    num_procedures=('icd_code', 'count'),
    unique_procedures=('procedure_description', lambda x: list(x.unique()))
).reset_index()

# Medications: Count number of prescribed drugs per hospitalization
med_summary = prescriptions.groupby('hadm_id').agg(
    num_medications=('drug', 'count'),
    unique_medications=('drug', lambda x: list(x.unique()))
).reset_index()

# Merge all features into a single dataset
dataset = features.merge(icu_summary, on='hadm_id', how='left', suffixes=None)
dataset = dataset.merge(diagnosis_summary, on='hadm_id', how='left', suffixes=None)
dataset = dataset.merge(procedure_summary, on='hadm_id', how='left', suffixes=None)
dataset = dataset.merge(med_summary, on='hadm_id', how='left', suffixes=None)

# Fill missing values with 0 or "None" where appropriate
# To discuss whether is it properly appropriate or not.
dataset.fillna({'icu_admissions': 0, 'icu_hours': 0, 'num_diagnoses': 0, 'num_procedures': 0, 'num_medications': 0}, inplace=True)
dataset.fillna({'unique_diagnoses': 'None', 'unique_procedures': 'None', 'unique_medications': 'None'}, inplace=True)

/tmp/ipykernel_5389/124748171.py:38: FutureWarning: Passing 'suffixes' as a <class 'NoneType'>, is not supported and may give unexpected results. Provide 'suffixes' as a tuple instead. In the future a 'TypeError' will be raised.
  dataset = features.merge(icu_summary, on='hadm_id', how='left', suffixes=None)
/tmp/ipykernel_5389/124748171.py:39: FutureWarning: Passing 'suffixes' as a <class 'NoneType'>, is not supported and may give unexpected results. Provide 'suffixes' as a tuple instead. In the future a 'TypeError' will be raised.
  dataset = dataset.merge(diagnosis_summary, on='hadm_id', how='left', suffixes=None)
/tmp/ipykernel_5389/124748171.py:40: FutureWarning: Passing 'suffixes' as a <class 'NoneType'>, is not supported and may give unexpected results. Provide 'suffixes' as a tuple instead. In the future a 'TypeError' will be raised.
  dataset = dataset.merge(procedure_summary, on='hadm_id', how='left', suffixes=None)
/tmp/ipykernel_5389/124748171.py:41: FutureWarning: Passing 

In [11]:
dataset.head(5)

,subject_id,hadm_id,age_at_event,gender,admission_type,discharge_location,icu_admissions,total_icu_hours,num_diagnoses,unique_diagnoses,num_procedures,unique_procedures,num_medications,unique_medications
0,10000032,22595853,52,F,URGENT,HOME,0.0,NaN,8.0,"[Portal hypertension, Other ascites, Cirrhosis...",1.0,[Percutaneous abdominal drainage],14.0,"[Furosemide, Ipratropium Bromide Neb, Potassiu..."
1,10000032,22841357,52,F,EW EMER.,HOME,0.0,NaN,8.0,[Unspecified viral hepatitis C with hepatic co...,1.0,[Percutaneous abdominal drainage],15.0,"[Furosemide, Rifaximin, Sodium Chloride 0.9% ..."
2,10000032,25742920,52,F,EW EMER.,HOSPICE,0.0,NaN,11.0,[Chronic hepatitis C without mention of hepati...,1.0,[Percutaneous abdominal drainage],28.0,"[Sodium Chloride 0.9% Flush, 0.9% Sodium Chlo..."
3,10000032,29079034,52,F,EW EMER.,HOME,1.0,9.0,14.0,"[Other iatrogenic hypotension, Chronic hepatit...",0.0,None,24.0,"[Bisacodyl, Senna, Calcium Carbonate, Raltegra..."
4,10000068,25022803,19,F,EU OBSERVATION,NaN,0.0,NaN,1.0,"[Alcohol abuse, unspecified]",1.0,[Other nonoperative respiratory measurements],0.0,None


In [5]:
dataset.loc[0, 'unique_diagnoses']

['Portal hypertension',
 'Other ascites',
 'Cirrhosis of liver without mention of alcohol',
 'Unspecified viral hepatitis C without hepatic coma',
 'Chronic airway obstruction, not elsewhere classified',
 'Bipolar disorder, unspecified',
 'Posttraumatic stress disorder',
 'Personal history of tobacco use']

In [ ]:
# ------------------------- STEP 5: SAVE & DISPLAY DATASET -------------------------

# Save the dataset to a CSV file for ML usage
dataset.to_csv("mimiciv_clinical_dataset.csv", index=False)

# Display the first few rows
import ace_tools as tools
tools.display_dataframe_to_user(name="MIMIC-IV Clinical Dataset", dataframe=dataset)
